# Capstone Project DB15-G009: AI-Assisted Custom Order Management (Toko Firaza)
**Notebook Eksperimen Tim Generative AI (Prioritas 1 – 4)**

Notebook ini dirancang untuk dijalankan di **Google Colab (GPU T4)** maupun lokal, mencakup:
1. **Prioritas 1:** Definisi Skema JSON & Validasi Pydantic (*API Contract*).
2. **Prioritas 2:** Pembuatan *Synthetic Dataset* & *Test Set* (1.000 sampel chat pelanggan).
3. **Prioritas 3:** Pelatihan & Evaluasi *Predictive AI* (Scikit-Learn TF-IDF + Calibrated LinearSVC).
4. **Prioritas 4:** Eksperimen *Generative AI* (Open-Source LLM Lokal / HuggingFace / Ollama + *Strict Few-Shot Prompting*).

In [ ]:
# 1. Install Dependencies di Google Colab
!pip install -q scikit-learn joblib pydantic fastapi uvicorn

In [ ]:
# 2. Jalankan Generator Synthetic Dataset, Training Model Klasifikasi, dan Evaluasi Pipeline
import os, sys
if os.path.exists('../src'):
    sys.path.insert(0, os.path.abspath('..'))

from src.generate_dataset import main as generate_dataset
from src.train_classifier import train_and_evaluate
from src.evaluate_pipeline import evaluate_end_to_end

generate_dataset()
train_and_evaluate()
evaluate_end_to_end()

In [ ]:
# 3. Uji Coba Interaktif Pesan Pelanggan Baru
import json
from src.train_classifier import OrderClassifier
from src.llm_extractor import LocalLLMOrderExtractor

clf = OrderClassifier().load()
ext = LocalLLMOrderExtractor()

sample_chat = (
    "Assalamualaikum kak, saya Farhan mau pesen hampers lebaran 3 paket wrn hijau emerald. "
    "Isinya sajadah custom uk dewasa bdr nama 'Keluarga Besar Sutrisno' pake font kaligrafi arab di pojok kanan bawah. "
    "Sekalian kartu ucapan selamat idul fitri dari Farhan ya min."
)

pred = clf.predict(sample_chat)
summary, used_fallback = ext.extract(sample_chat, pred.category, generate_greeting_card=True)

print('=== PREDIKSI KATEGORI ===')
print(json.dumps(pred.model_dump(), indent=2, ensure_ascii=False))
print('\n=== HASIL EKSTRAKSI ORDER & KARTU UCAPAN ===')
print(json.dumps(summary.model_dump(), indent=2, ensure_ascii=False))